# Macaron drawing style test
Draws a set of dishes in two slightly cartoon styles with a model that follows detailed descriptions closely. Only a contact sheet is saved.

In [ ]:
!pip -q install -U bitsandbytes 2>&1 | tail -1
!pip -q install --no-deps rembg pymatting onnxruntime flatbuffers coloredlogs humanfriendly 2>&1 | tail -2
import diffusers, transformers; print("diffusers", diffusers.__version__, "transformers", transformers.__version__)

In [ ]:
import torch, time, numpy as np
from PIL import Image, ImageDraw, ImageFont
from diffusers import FluxPipeline, FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TfBnb
from rembg import remove, new_session
REPO = "black-forest-labs/FLUX.1-schnell"
def load(dt):
    tr = FluxTransformer2DModel.from_pretrained(REPO, subfolder="transformer", torch_dtype=dt,
        quantization_config=DiffBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dt))
    t5 = T5EncoderModel.from_pretrained(REPO, subfolder="text_encoder_2", torch_dtype=dt, quantization_config=TfBnb(load_in_8bit=True))
    p = FluxPipeline.from_pretrained(REPO, transformer=tr, text_encoder_2=t5, torch_dtype=dt)
    p.enable_model_cpu_offload(); return p
CUT = new_session("isnet-general-use")
DT = torch.float16
pipe = load(DT)
def gen2(prompt, seed, size=768):
    g = torch.Generator("cpu").manual_seed(seed)
    return pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0, max_sequence_length=256, height=size, width=size, generator=g).images[0]
t = time.time(); test = gen2("a bowl of white rice, illustration", 1)
if np.asarray(test).mean() < 3:
    print("fp16 gave a blank picture, switching to bfloat16"); del pipe; torch.cuda.empty_cache(); DT = torch.bfloat16; pipe = load(DT); test = gen2("a bowl of white rice, illustration", 1)
print("first picture", round(time.time() - t), "s, mean", np.asarray(test).mean())
def cut(img):
    o = remove(img, session=CUT).convert("RGBA"); bb = o.getbbox()
    if bb: o = o.crop(bb)
    o.thumbnail((300, 300), Image.LANCZOS); return o

In [ ]:
DISHES = {
 "Chicken rice, roasted": "a plate of Malaysian roasted chicken rice. On the left, six neat slices of boneless roasted chicken meat with glossy golden-brown skin, cut across into strips and laid in a row on a few cucumber slices, with a little dark soy sauce under them. On the right, a smooth dome of pale yellow chicken rice. A tiny saucer of bright red chilli sauce at the back",
 "Chicken rice, steamed": "a plate of Malaysian Hainanese steamed chicken rice. Six neat slices of boneless poached chicken with smooth pale white skin, laid in a row on cucumber slices, drizzled with light soy sauce and sprinkled with spring onion and coriander. A smooth dome of pale yellow rice beside it and a tiny saucer of red chilli sauce",
 "Fried chicken drumstick": "one golden crispy fried chicken drumstick with a rough craggy coating, lying on a small white plate",
 "Wantan mee, dry": "a plate of Malaysian dry wantan mee. A heap of thin springy egg noodles tossed in glossy dark soy sauce, topped with five thin slices of red-edged char siu barbecued pork and a few stalks of green choy sum. Beside the plate, a small bowl of clear soup with three silky boiled wontons",
 "Char kuey teow": "a plate of Penang char kuey teow. Wide flat rice noodles stir-fried until smoky dark brown, mixed with four prawns, small cockles, bean sprouts, green chives and small bits of scrambled egg mixed through the noodles",
 "Fried wantan": "a small plate of five golden deep-fried wontons. Each wonton is a round puffy pouch with crinkly, ruffled, frilly crispy edges like a flower. A small dish of red sweet chilli sauce beside them",
 "Kaya toast set": "Malaysian kaya toast set. Two thin slices of crisp toasted bread cut into rectangles, sandwiched with a layer of green kaya jam and a thick cold slice of yellow butter. Beside them, a small white bowl holding two runny soft-boiled eggs, cracked open, with dark soy sauce. A cup of black kopi coffee",
 "Mee goreng mamak": "a plate of Malaysian mee goreng mamak. Thick yellow noodles stir-fried in a red-orange spicy sauce, mixed with cubes of fried tofu, potato pieces, bean sprouts, green choy sum and bits of scrambled egg through the noodles. A lime half on the side",
 "Nasi lemak biasa": "Malaysian nasi lemak on a piece of banana leaf. A dome of white coconut rice, a spoon of dark red sambal, a small heap of crispy fried anchovies, a small heap of roasted peanuts, four cucumber slices and half a hard-boiled egg showing the yellow yolk",
 "Satay ayam": "Malaysian chicken satay. Six bamboo skewers of grilled chicken pieces with caramelised charred edges, a small bowl of thick peanut sauce, cubes of cucumber and red onion, and a few rice cake cubes",
 "Roti canai kosong": "Malaysian roti canai. Two round flaky golden flatbreads with crispy layered edges, loosely folded on a plate, with a small bowl of yellow dhal curry",
 "Curry laksa": "a bowl of Malaysian curry laksa. Orange coconut curry soup with yellow noodles, three tofu puffs, four prawns, half a boiled egg, bean sprouts and fresh mint leaves"
}
STYLES = {
 "A2 soft cartoon": "a cute hand-drawn food illustration of {d}. Soft painted cartoon style, simple clean shapes, gentle dark outlines, soft cel shading, warm cheerful colours, slightly stylized, like a picture in a children's cookbook, three-quarter view, centered, plain white background",
 "A3 painted lighter": "a soft digital painting illustration of {d}. Slightly stylized cartoon look, simplified shapes, soft brush strokes, gentle outlines, light flat shading, warm cheerful colours, three-quarter view, centered, plain white background"
}
res = {}
for d, desc in DISHES.items():
    for s, p in STYLES.items():
        t = time.time(); res[(d, s)] = cut(gen2(p.format(d=desc), 17)); print(d, "|", s, round(time.time() - t), "s")

In [ ]:
C = 230; names = list(DISHES); sts = list(STYLES)
sheet = Image.new("RGB", (len(names) * C, len(sts) * (C + 10) + 40), "#fff0f4"); dr = ImageDraw.Draw(sheet)
try: f = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 13)
except Exception: f = ImageFont.load_default()
for i, d in enumerate(names):
    dr.text((i * C + 6, 8), d[:28], fill="#7a2e4d", font=f)
    for j, s in enumerate(sts):
        im = res[(d, s)].copy(); im.thumbnail((C - 16, C - 16)); sheet.paste(im, (i * C + (C - im.width) // 2, 34 + j * (C + 10) + (C - im.height) // 2), im)
for j, s in enumerate(sts): dr.text((6, 34 + j * (C + 10) + C - 14), s, fill="#7a2e4d", font=f)
# two rows of 6 is easier to see on a phone
W = 6; out = Image.new("RGB", (W * C, 2 * len(sts) * (C + 10) + 60), "#fff0f4"); od = ImageDraw.Draw(out)
for i, d in enumerate(names):
    r, c = divmod(i, W); y0 = r * len(sts) * (C + 10) + r * 30
    od.text((c * C + 6, y0 + 6), d[:28], fill="#7a2e4d", font=f)
    for j, s in enumerate(sts):
        im = res[(d, s)].copy(); im.thumbnail((C - 20, C - 20)); out.paste(im, (c * C + (C - im.width) // 2, y0 + 26 + j * (C + 10)), im)
out.save("/kaggle/working/contact_styles.jpg", quality=88); print("sheet saved", out.size, "rows: top =", sts[0], ", bottom =", sts[1])